# 04 – Evaluation Analysis & Results Comparison

Investigate model evaluation results from the run registry and compare different training runs.

**Pre-requisites**: Run `02_training.ipynb` first, then evaluate your models:
```bash
python -m src.evaluate --training-dir outputs/20260420_170147 --split test
```

Then this notebook will:
1. Load evaluation results from registry
2. Display metrics (Recall@K, NDCG@K, HR@K)
3. Compare multiple runs
4. Visualize performance trends

## 1. Setup & Load Registry

In [1]:
import sys
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure src/ is in Python path
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.run_registry import RunRegistry

print(f"✓ Project root: {project_root}")
print(f"✓ All imports successful")

✓ Project root: c:\Users\elisk\Desktop\2024-25\Diplomka\Github\Diplomov-pr-ce
✓ All imports successful


In [2]:
# Load the run registry
registry = RunRegistry()

# Get all runs
all_runs = registry.get_runs()
print(f"Total runs in registry: {len(all_runs)}")

# Get runs with evaluation results
evaluated_runs = []
for run_id in all_runs:
    run_info = registry.get_run_info(run_id)
    if 'evaluate' in run_info['stages']:
        stage_info = run_info['stages']['evaluate']
        if stage_info['status'] == 'completed':
            evaluated_runs.append(run_id)

print(f"\nRuns with evaluation results: {len(evaluated_runs)}")

if evaluated_runs:
    print("\nEvaluated runs:")
    for run_id in evaluated_runs[:5]:  # Show first 5
        print(f"  - {run_id}")
else:
    print("\n⚠️  No evaluation results yet. Run:")
    print("   python -m src.evaluate --training-dir outputs/RUNID --split test")

Total runs in registry: 0

Runs with evaluation results: 0

⚠️  No evaluation results yet. Run:
   python -m src.evaluate --training-dir outputs/RUNID --split test


## 2. Explore a Specific Run

In [3]:
# Select a run to analyze
if evaluated_runs:
    selected_run = evaluated_runs[0]  # Latest evaluated run
    print(f"Selected run: {selected_run}")
else:
    print("❌ No evaluated runs available")
    selected_run = None

# Get run details
if selected_run:
    run_info = registry.get_run_info(selected_run)
    
    print(f"\n" + "="*80)
    print(f"RUN: {selected_run}")
    print("="*80)
    print(f"Created: {run_info['created']}")
    
    # Show all stages
    print(f"\nStages:")
    for stage, stage_info in run_info['stages'].items():
        status = stage_info['status']
        status_symbol = {'completed': '✓', 'pending': '⏳', 'failed': '✗'}.get(status, '?')
        print(f"  {status_symbol} {stage}: {status}")
        
        # Show metadata
        if stage_info['metadata']:
            for key, value in list(stage_info['metadata'].items())[:5]:  # First 5 items
                if isinstance(value, (int, float)):
                    print(f"      - {key}: {value}")

❌ No evaluated runs available


In [ ]:
# Load metrics from summary.json (preferred) or old evaluation files
if selected_run:
    eval_dir = project_root / "outputs" / selected_run
    summary_file = eval_dir / "summary.json"
    
    if summary_file.exists():
        with open(summary_file, 'r') as f:
            summary = json.load(f)
        
        print(f"\n" + "="*80)
        print(f"TRAINING METRICS (from summary.json)")
        print("="*80)
        
        # Show ELSA metrics
        if 'ranking_metrics_elsa' in summary:
            print("\n📊 ELSA (baseline - dense latents, no SAE):")
            for k in [5, 10, 20]:
                recall = summary['ranking_metrics_elsa'].get(f'Recall@{k}', 0)
                ndcg = summary['ranking_metrics_elsa'].get(f'NDCG@{k}', 0)
                print(f"  @{k:2d}  Recall: {recall:.4f}  NDCG: {ndcg:.4f}")
        
        # Show SAE+ELSA metrics
        if 'ranking_metrics_sae' in summary:
            print("\n⚡ SAE + ELSA (sparse features):")
            for k in [5, 10, 20]:
                recall = summary['ranking_metrics_sae'].get(f'Recall@{k}', 0)
                ndcg = summary['ranking_metrics_sae'].get(f'NDCG@{k}', 0)
                print(f"  @{k:2d}  Recall: {recall:.4f}  NDCG: {ndcg:.4f}")
        
        # Show comparison if available
        if 'ranking_metrics_elsa' in summary and 'ranking_metrics_sae' in summary:
            print("\n⚖️  COMPARISON (SAE vs ELSA):")
            for k in [5, 10, 20]:
                elsa_recall = summary['ranking_metrics_elsa'].get(f'Recall@{k}', 0)
                sae_recall = summary['ranking_metrics_sae'].get(f'Recall@{k}', 0)
                diff = (sae_recall - elsa_recall) / (elsa_recall + 1e-8) * 100
                symbol = "↑" if diff > 0 else "↓" if diff < 0 else "→"
                print(f"  Recall@{k:2d}:  {diff:+6.2f}% {symbol}")
    else:
        # Fallback to old evaluation files
        eval_file = eval_dir / "evaluation_test.json"
        if eval_file.exists():
            with open(eval_file, 'r') as f:
                eval_results = json.load(f)
            
            print(f"\n" + "="*80)
            print(f"EVALUATION METRICS (from evaluation_test.json)")
            print("="*80)
            print(f"\nMetrics:")
            
            metrics = eval_results.get('metrics', {})
            for metric_name in sorted(metrics.keys()):
                value = metrics[metric_name]
                print(f"  {metric_name:.<30} {value:.4f}")
        else:
            print(f"❌ Neither summary.json nor evaluation_test.json found")
            json_files = list(eval_dir.glob("*.json"))
            if json_files:
                print(f"   Available: {[f.name for f in json_files[:5]]}")


## 3. Visualize Metrics for Selected Run

In [ ]:
# ELSA vs SAE+ELSA Comparison Visualization
if selected_run and 'summary' in locals() and 'ranking_metrics_elsa' in summary:
    print("\n" + "="*80)
    print("MODEL COMPARISON: ELSA vs SAE+ELSA")
    print("="*80)
    
    # Extract metrics for each k value
    comparison_data = []
    for k in [5, 10, 20]:
        elsa_recall = summary['ranking_metrics_elsa'].get(f'Recall@{k}', 0)
        sae_recall = summary['ranking_metrics_sae'].get(f'Recall@{k}', 0)
        elsa_ndcg = summary['ranking_metrics_elsa'].get(f'NDCG@{k}', 0)
        sae_ndcg = summary['ranking_metrics_sae'].get(f'NDCG@{k}', 0)
        
        comparison_data.append({
            'k': k,
            'ELSA_Recall': elsa_recall,
            'SAE_Recall': sae_recall,
            'Recall_Diff%': (sae_recall - elsa_recall) / (elsa_recall + 1e-8) * 100,
            'ELSA_NDCG': elsa_ndcg,
            'SAE_NDCG': sae_ndcg,
            'NDCG_Diff%': (sae_ndcg - elsa_ndcg) / (elsa_ndcg + 1e-8) * 100,
        })
    
    comp_df = pd.DataFrame(comparison_data)
    
    # Display as table
    pd.set_option('display.float_format', '{:.4f}'.format)
    pd.set_option('display.max_columns', None)
    print("\n", comp_df.to_string(index=False))
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Recall comparison
    x = comp_df['k']
    axes[0].plot(x, comp_df['ELSA_Recall'], 'o-', label='ELSA', linewidth=2, markersize=8)
    axes[0].plot(x, comp_df['SAE_Recall'], 's-', label='SAE+ELSA', linewidth=2, markersize=8)
    axes[0].set_xlabel('k (cutoff)', fontsize=11)
    axes[0].set_ylabel('Recall@k', fontsize=11)
    axes[0].set_title('Recall Comparison: ELSA vs SAE+ELSA', fontsize=12, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(alpha=0.3)
    axes[0].set_xticks(x)
    
    # NDCG comparison
    axes[1].plot(x, comp_df['ELSA_NDCG'], 'o-', label='ELSA', linewidth=2, markersize=8)
    axes[1].plot(x, comp_df['SAE_NDCG'], 's-', label='SAE+ELSA', linewidth=2, markersize=8)
    axes[1].set_xlabel('k (cutoff)', fontsize=11)
    axes[1].set_ylabel('NDCG@k', fontsize=11)
    axes[1].set_title('NDCG Comparison: ELSA vs SAE+ELSA', fontsize=12, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(alpha=0.3)
    axes[1].set_xticks(x)
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print("\n📊 SUMMARY:")
    avg_recall_diff = comp_df['Recall_Diff%'].mean()
    avg_ndcg_diff = comp_df['NDCG_Diff%'].mean()
    print(f"  Avg Recall change: {avg_recall_diff:+.2f}%")
    print(f"  Avg NDCG change:   {avg_ndcg_diff:+.2f}%")
    
    if 'model_size_elsa_mb' in summary and 'model_size_sae_mb' in summary:
        elsa_size = summary.get('model_size_elsa_mb', 0)
        sae_size = summary.get('model_size_sae_mb', 0)
        compression = (1 - sae_size / (elsa_size + sae_size)) * 100
        print(f"  ELSA size: {elsa_size:.1f}MB  |  SAE size: {sae_size:.1f}MB")
        print(f"  Compression gain: {compression:.1f}% (toward interpretability)")


## 4. Compare Multiple Runs

In [6]:
# Collect metrics from all evaluated runs
comparison_data = []

for run_id in evaluated_runs:
    eval_dir = project_root / "outputs" / run_id
    eval_file = eval_dir / "evaluation_test.json"
    
    if eval_file.exists():
        with open(eval_file, 'r') as f:
            eval_results = json.load(f)
        
        metrics = eval_results.get('metrics', {})
        row = {'run_id': run_id}
        row.update(metrics)
        comparison_data.append(row)

if comparison_data:
    # Create comparison DataFrame
    df_comparison = pd.DataFrame(comparison_data)
    
    print(f"\n" + "="*80)
    print(f"COMPARISON OF {len(df_comparison)} RUNS")
    print("="*80)
    
    # Show table
    display_cols = ['run_id'] + [c for c in df_comparison.columns if c != 'run_id'][:5]
    print(df_comparison[display_cols].to_string())
    
    print(f"\n\nSummary Statistics:")
    print(df_comparison[[c for c in df_comparison.columns if c != 'run_id']].describe())
else:
    print("No comparison data available")

No comparison data available


## 5. Analyze Best Performance

In [7]:
# Find best run by each metric
if comparison_data:
    print("\n" + "="*80)
    print("BEST PERFORMANCE BY METRIC")
    print("="*80)
    
    metric_cols = [c for c in df_comparison.columns if c != 'run_id']
    
    for metric in sorted(metric_cols):
        if metric in df_comparison.columns:
            best_idx = df_comparison[metric].idxmax()
            best_run = df_comparison.loc[best_idx, 'run_id']
            best_value = df_comparison.loc[best_idx, metric]
            
            print(f"\n{metric:.<40} {best_value:.4f}")
            print(f"  Run: {best_run}")

## 6. Registry Summary

In [8]:
# Show complete registry summary
print("\n")
registry.print_summary()




RUN REGISTRY SUMMARY
No runs registered yet.
